In [1]:
# %%
# =============================================================================
# NOTEBOOK: 03_longtext_strategies_D_TDC.ipynb  (FULL RUN)
# Paper D (TDC) — "simplicity wins": full-scale comparison of long-document
# strategies for 10-K bankruptcy signals.
#
# Pilot showed: summarization matched plain truncation at ~5x the cost →
# EXCLUDED from the full run and reported as a finding. Full run compares
# truncation / extraction / chunking on all 222 firms × 3 tiers.
#
# Data: text-based bankruptcy dataset, Mendeley DOI 10.17632/stf3kg7fw3
# =============================================================================
import os, json, time, re, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

try:
    import tiktoken
    ENC = tiktoken.get_encoding("cl100k_base")
    def head_tok(s, n): return ENC.decode(ENC.encode(s)[:n])
    def chunks_tok(s, n):
        t = ENC.encode(s); return [ENC.decode(t[i:i+n]) for i in range(0, len(t), n)]
except ImportError:
    def head_tok(s, n):
        w = s.split(); return " ".join(w[:int(n/1.3)])
    def chunks_tok(s, n):
        w = s.split(); step = int(n/1.3)
        return [" ".join(w[i:i+step]) for i in range(0, len(w), step)]

ROOT = Path.cwd()
if ROOT.name in {"notebooks", "paperA", "paperB", "paperC", "paperD"}:
    ROOT = ROOT.parents[0] if ROOT.name == "notebooks" else ROOT.parents[1]
TEXT_DIR = ROOT / "data" / "raw" / "text_bankruptcy"
INFER = ROOT / "artifacts" / "inference"
TAB = ROOT / "artifacts" / "tables"
for p in (INFER, TAB):
    p.mkdir(parents=True, exist_ok=True)


def rel(p):
    try: return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError: return Path(p).name


load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODELS = {
    "weak":   {"name": "gpt-4o-mini",  "in": 0.15, "cached_in": 0.075, "out": 0.60},
    "mid":    {"name": "gpt-5.4-mini", "in": 0.75, "cached_in": 0.075, "out": 4.50},
    "strong": {"name": "gpt-5.4",      "in": 2.50, "cached_in": 0.25,  "out": 15.00},
}
SEED = 42; np.random.seed(SEED)
TRUNC_TOKENS = 8000; CHUNK_TOKENS = 8000
STRATEGIES = ["truncation", "extraction", "chunking"]      # summarization excluded

# --- Full pooled dataset (222) -----------------------------------------------
Xtr = pd.read_csv(TEXT_DIR / "NUM10K_X_train_2021June.csv")
Xte = pd.read_csv(TEXT_DIR / "NUM10K_X_test_2021June.csv")
ytr = pd.read_csv(TEXT_DIR / "NUM10K_y_train_2021June.csv")["Bankruptcy"]
yte = pd.read_csv(TEXT_DIR / "NUM10K_y_test_2021June.csv")["Bankruptcy"]
Xtr["Bankruptcy"] = ytr.values; Xte["Bankruptcy"] = yte.values
df = pd.concat([Xtr, Xte], ignore_index=True).reset_index(drop=True)
df["firm_id"] = df.index
df.to_parquet(INFER / "D_full_firms.parquet", index=False)
print(f"Full: {len(df)} firms, {int(df.Bankruptcy.sum())} bankrupt")

RISK_KEYWORDS = ["going concern","substantial doubt","liquidity","default","covenant",
    "bankruptcy","insolvency","litigation","impairment","deficit","material weakness",
    "unable to continue","debt maturity","restructuring"]

def get_text(row):
    return str(row.get("MDandA","")) + "\n\n" + str(row.get("RiskFactors",""))

def segments_for(strategy, text):
    if strategy == "truncation":
        return [head_tok(text, TRUNC_TOKENS)]
    if strategy == "extraction":
        sents = re.split(r"(?<=[.!?])\s+", text)
        kept = [s for s in sents if any(k in s.lower() for k in RISK_KEYWORDS)]
        return [head_tok(" ".join(kept) if kept else head_tok(text, 2000), TRUNC_TOKENS)]
    if strategy == "chunking":
        return chunks_tok(text, CHUNK_TOKENS)
    raise ValueError(strategy)

JUDGE_SYS = ("You are a credit analyst. Based on the provided 10-K excerpt(s), assess "
    "the firm's bankruptcy risk. Respond ONLY with JSON: "
    '{"verdict":"healthy"|"at risk","risk_score":0-100,"rationale":"one sentence"}')

def parse_judge(text):
    t = text.strip()
    if t.startswith("```"): t = t.strip("`"); t = t[t.find("{"):]
    a, b = t.find("{"), t.rfind("}")
    try:
        o = json.loads(t[a:b+1])
        v = "at risk" if "risk" in str(o.get("verdict","")).lower() else "healthy"
        return v, float(o.get("risk_score", 50))
    except Exception:
        return ("at risk" if "at risk" in t.lower() else "healthy"), 50.0

def run_full(strategy, tier):
    model = MODELS[tier]["name"]
    firms = pd.read_parquet(INFER / "D_full_firms.parquet")
    cache_fp = INFER / f"D_full_{strategy}_{tier}.jsonl"
    done = set()
    if cache_fp.exists():
        for l in cache_fp.read_text(encoding="utf-8").splitlines():
            done.add(json.loads(l)["firm_id"])
    with cache_fp.open("a", encoding="utf-8") as f:
        for _, row in tqdm(firms.iterrows(), total=len(firms), desc=f"{strategy}/{tier}"):
            fid = int(row["firm_id"])
            if fid in done: continue
            segs = segments_for(strategy, get_text(row))
            outs, pin, pout, cach = [], 0, 0, 0
            try:
                for seg in segs:
                    r = client.chat.completions.create(
                        model=model, messages=[{"role":"system","content":JUDGE_SYS},
                                                {"role":"user","content":seg}],
                        temperature=0.0, seed=SEED,
                        response_format={"type":"json_object"})
                    v, s = parse_judge(r.choices[0].message.content)
                    outs.append((v, s)); u = r.usage
                    pin += u.prompt_tokens; pout += u.completion_tokens
                    cach += getattr(getattr(u,"prompt_tokens_details",None),"cached_tokens",0) or 0
            except Exception as e:
                print(f"  firm {fid}: {type(e).__name__}: {e}"); time.sleep(3); continue
            scores = [s for _, s in outs]; verds = [v for v, _ in outs]
            rec = {"firm_id": fid, "strategy": strategy, "tier": tier, "model": model,
                   "bankrupt": int(row["Bankruptcy"]),
                   "verdict": "at risk" if verds.count("at risk") >= len(verds)/2 else "healthy",
                   "risk_score": float(np.mean(scores)), "n_segments": len(outs),
                   "prompt_tokens": pin, "completion_tokens": pout, "cached_tokens": cach}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            time.sleep(0.1)

for strat in STRATEGIES:
    for tier in ["weak", "mid", "strong"]:
        run_full(strat, tier)
print("Full run complete.")

Full: 222 firms, 111 bankrupt


chunking/strong: 100%|██████████| 222/222 [33:26<00:00,  9.04s/it]

Full run complete.


In [2]:
# %%
# =============================================================================
# Cell 2. Confirm the ranking with bootstrap CIs on 222 firms, and contrast
# every text strategy with the numeric baseline (0.862). Tests the paper's two
# claims: (i) elaborate strategies don't beat truncation; (ii) text ≈ numeric.
# =============================================================================
from sklearn.metrics import roc_auc_score, accuracy_score

def boot_auc_ci(y, s, n=2000, alpha=0.05):
    y, s = np.array(y), np.array(s); idx = np.arange(len(y)); aucs = []
    for _ in range(n):
        b = np.random.choice(idx, len(idx), replace=True)
        if len(set(y[b])) < 2: continue
        aucs.append(roc_auc_score(y[b], s[b]))
    return np.mean(aucs), np.percentile(aucs, [100*alpha/2, 100*(1-alpha/2)])

rows = []
for strat in STRATEGIES:
    for tier in ["weak", "mid", "strong"]:
        fp = INFER / f"D_full_{strat}_{tier}.jsonl"
        if not fp.exists(): continue
        d = pd.DataFrame([json.loads(l) for l in fp.read_text(encoding="utf-8").splitlines()])
        if not len(d): continue
        m = MODELS[tier]
        cost = ((d.prompt_tokens.sum()-d.cached_tokens.sum())*m["in"]
                + d.cached_tokens.sum()*m["cached_in"] + d.completion_tokens.sum()*m["out"])/1e6
        auc, ci = boot_auc_ci(d.bankrupt.values, d.risk_score.values)
        acc = accuracy_score(d.bankrupt.values, (d.verdict=="at risk").astype(int).values)
        rows.append({"strategy": strat, "model": m["name"], "n": len(d),
                     "ROC_AUC": auc, "ci_low": ci[0], "ci_high": ci[1],
                     "accuracy": acc, "cost_usd": cost})

res = pd.DataFrame(rows)
print("── FULL run: strategy × tier (bootstrap 95% CI on 222 firms) ──")
for _, r in res.iterrows():
    print(f"  {r.strategy:12s} {r.model:14s}: AUC {r.ROC_AUC:.3f} "
          f"[{r.ci_low:.3f}, {r.ci_high:.3f}]  acc {r.accuracy:.3f}  ${r.cost_usd:.3f}")

print(f"\n  Numeric baseline (02): ROC-AUC 0.862 ±0.050")
print("  Claim (i): if truncation's CI overlaps chunking's, elaborate ≠ better.")
print("  Claim (ii): if text CIs include 0.862, text does not beat numeric.")

# best text strategy per tier vs numeric
print("\n── Best text strategy per tier ──")
for tier_name in res.model.unique():
    sub = res[res.model == tier_name]
    best = sub.loc[sub.ROC_AUC.idxmax()]
    beats = "overlaps" if best.ci_low <= 0.862 <= best.ci_high else \
            ("ABOVE" if best.ci_low > 0.862 else "below")
    print(f"  {tier_name:14s}: best={best.strategy} AUC {best.ROC_AUC:.3f} "
          f"→ vs numeric 0.862: {beats}")

res.round(4).to_csv(TAB / "D_full_strategy_comparison.csv", index=False)
print(f"\n  → saved: {rel(TAB / 'D_full_strategy_comparison.csv')}")

── FULL run: strategy × tier (bootstrap 95% CI on 222 firms) ──
  truncation   gpt-4o-mini   : AUC 0.825 [0.771, 0.873]  acc 0.599  $0.247
  truncation   gpt-5.4-mini  : AUC 0.894 [0.851, 0.932]  acc 0.707  $1.261
  truncation   gpt-5.4       : AUC 0.910 [0.870, 0.944]  acc 0.833  $4.225
  extraction   gpt-4o-mini   : AUC 0.825 [0.773, 0.873]  acc 0.523  $0.114
  extraction   gpt-5.4-mini  : AUC 0.873 [0.826, 0.916]  acc 0.608  $0.593
  extraction   gpt-5.4       : AUC 0.893 [0.845, 0.932]  acc 0.716  $2.012
  chunking     gpt-4o-mini   : AUC 0.856 [0.802, 0.903]  acc 0.509  $1.045
  chunking     gpt-5.4-mini  : AUC 0.875 [0.824, 0.919]  acc 0.635  $5.271
  chunking     gpt-5.4       : AUC 0.898 [0.852, 0.938]  acc 0.739  $16.962

  Numeric baseline (02): ROC-AUC 0.862 ±0.050
  Claim (i): if truncation's CI overlaps chunking's, elaborate ≠ better.
  Claim (ii): if text CIs include 0.862, text does not beat numeric.

── Best text strategy per tier ──
  gpt-4o-mini   : best=chunking AUC 